<a href="https://colab.research.google.com/github/hcarpenter30/IS4487_my_portfolio/blob/main/carpenter_henry_assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [8]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('cleaned_airbnb_data_7.csv')

# Preview the dataset
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,...,price_log,minimum_nights_minmax,number_of_reviews_zscore,availability_category,price_per_accommodate,is_long_term_stay,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,360,https://www.airbnb.com/rooms/360,20260630053644,2026-06-30,city scrape,Denver’s Peaceful Oasis for Travel Nurses & No...,Enjoy the famous Colorado weather and unplug i...,https://a0.muscache.com/pictures/monet/Select-...,666,https://www.airbnb.com/users/show/666,...,4.676095,0.056112,0.907760,High,53.175000,0,1,0,0,0
1,364,https://www.airbnb.com/rooms/364,20260630053644,2026-07-02,previous scrape,Lodo / RiNo LOFT via airport train,"Modern 1,000 square foot loft in the heart of ...",https://a0.muscache.com/pictures/11766413/a2c5...,783,https://www.airbnb.com/users/show/783,...,3.507058,0.368737,0.120787,High,10.783333,1,1,0,0,0
2,592,https://www.airbnb.com/rooms/592,20260630053644,2026-06-30,city scrape,private,This room is in the basement. It does not hav...,https://a0.muscache.com/pictures/ba522ff9-84c9...,933,https://www.airbnb.com/users/show/933,...,4.394449,0.002004,1.185983,Medium,40.000000,0,0,0,1,0
3,686,https://www.airbnb.com/rooms/686,20260630053644,2026-06-30,city scrape,Alexandra's Queen Bed Room Long-term Rental Only,Thanks for visiting my Queen Bed Room for LONG...,https://a0.muscache.com/pictures/108112/e6d5d3...,990,https://www.airbnb.com/users/show/990,...,3.812645,0.058116,1.432408,Medium,44.270000,0,0,0,1,0
4,1940,https://www.airbnb.com/rooms/1940,20260630053644,2026-06-30,city scrape,"Baker Studio Pet-friendly, Walkable, Patio",Private studio retreat in a charming historic ...,https://a0.muscache.com/pictures/miso/Hosting-...,2150,https://www.airbnb.com/users/show/2150,...,5.261238,0.000000,1.639088,High,95.860000,0,1,0,0,0


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. The business information included in the dataset is the price log of the listings, the min and max nights, price per accomodate, etc. There are a lot of useful variables that can help businesses.

2. In the dataset, there are 4,939 rows and 96 total columns. A pretty large amount of columns and rows.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [10]:
# Columns to drop if they exist in the dataset
columns_to_drop = ['post_id', 'title', 'descr', 'details', 'address']

# Drop only the columns that actually exist in df
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

# Print the remaining columns and shape
print(f"Dropped columns: {existing_cols_to_drop}")
print(f"New shape: {df.shape}")

Dropped columns: []
New shape: (4939, 96)


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. The columns I dropped were post_id, title, description, details, and address. These variables just arent too important for predicting the target variable.

2. The risks associated with including them in the model is the model not being accurate since these variables don't tell us much or help us predict the target.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [11]:
# Select numeric columns
numeric_df = df.select_dtypes(include=['number'])

# Calculate correlation with the target variable 'price'
correlations = numeric_df.corr()['price'].sort_values(ascending=False)

# Display top 10 positive and top 10 negative correlations with price
print("Top 10 positive correlations with price:")
print(correlations.head(11)) # including price itself
print("\nTop 10 negative correlations with price:")
print(correlations.tail(10))

Top 10 positive correlations with price:
price_quote_price_per_night    1.000000
price                          1.000000
price_per_accommodate          0.985279
price_quote_total_price        0.970049
bedrooms                       0.263619
bathrooms                      0.261782
price_log                      0.259321
estimated_revenue_l365d        0.203951
accommodates                   0.067047
beds                           0.056979
hosts_time_as_host_years       0.037121
Name: price, dtype: float64

Top 10 negative correlations with price:
scrape_id                      NaN
host_response_rate             NaN
host_acceptance_rate           NaN
host_thumbnail_url             NaN
host_neighbourhood             NaN
host_total_listings_count      NaN
host_verifications             NaN
neighbourhood                  NaN
neighbourhood_group_cleansed   NaN
instant_bookable               NaN
Name: price, dtype: float64


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The variables with the strongest positive correlation to price are bedrooms (0.26), bathrooms (0.26), price_log (0.26), and estimated_revenue (0.20).

2. Features like bedrooms, bathrooms, accommodates, beds, and other property capacity characteristics will be highly useful predictors for the regression model.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [12]:
# 1. Define target variable (y)
y = df['price']

# 2. Select numeric columns for regression predictors
numeric_features = df.select_dtypes(include=['number'])

# 3. Drop 'price' and any columns derived directly from price to prevent leaking the target
# Also drop columns containing mostly NaNs or ID columns
columns_to_exclude = [
    'price', 'price_log', 'price_per_accommodate',
    'price_quote_price_per_night', 'price_quote_total_price',
    'id', 'scrape_id', 'host_id'
]

# Filter out exclusions that exist in numeric_features
columns_to_exclude = [col for col in columns_to_exclude if col in numeric_features.columns]

X = numeric_features.drop(columns=columns_to_exclude)

# Drop columns containing entirely NaN values if any remain
X = X.dropna(axis=1, how='all')

# Fill remaining NaNs with column median to ensure regression runs smoothly
X = X.fillna(X.median())

print("Target (y) shape:", y.shape)
print("Predictors (X) shape:", X.shape)

Target (y) shape: (4939,)
Predictors (X) shape: (4939, 50)


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. We are using 50 different features including bedrooms, bathrooms, accommodates, beds, historical review metrics, host characteristics, and binary dummy variables.

2. This is a regression problem because the target variable price is a continuous numerical value. Classification is used when the target variable are categories or classes, not something like price.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [13]:
from sklearn.model_selection import train_test_split

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training features shape: (3951, 50)
Testing features shape: (988, 50)
Training target shape: (3951,)
Testing target shape: (988,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [14]:
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Fit the model
model.fit(X_train, y_train)

# Predict prices for the test set
y_pred = model.predict(X_test)

print("Model training completed and predictions generated on test set.")

Model training completed and predictions generated on test set.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [15]:
from sklearn.metrics import mean_squared_error, r2_score

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)

# Calculate R-squared (R2) score
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

Mean Squared Error (MSE): 88037.67
R-squared (R2) Score: 0.0786


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response:
1. The R-squared score is 0.078. This shows the linear regression model only explains about 7.86% of the variation in Airbnb prices. This basically tells us that the current set of features does not capture most of the price variance.

2. The Mean Squared Error is quite large at 88,037. This indicates that our average predictions deviate from the true prices. To improve it, we could remove extreme outliers in pricing or try other methods like Decision Trees or Gradient Boosting.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [16]:
import pandas as pd

# Create a DataFrame of features and their coefficients
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Absolute_Coefficient': abs(model.coef_)
})

# Sort the features by absolute impact (highest to lowest)
coefficients_df = coefficients_df.sort_values(by='Absolute_Coefficient', ascending=False).drop(columns=['Absolute_Coefficient'])

# Display the table
coefficients_df

,Feature,Coefficient
30,estimated_revenue_l365d,5.396354e-03
29,estimated_occupancy_l365d,2.282187e-06
24,number_of_reviews,1.565741e-06
19,maximum_nights_avg_ntm,1.237156e-06
16,minimum_maximum_nights,1.225777e-06
13,maximum_nights,1.225022e-06
17,maximum_maximum_nights,1.225022e-06
25,number_of_reviews_ltm,4.262754e-07
28,number_of_reviews_ly,4.013238e-07
15,maximum_minimum_nights,-3.160088e-07


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. The feature with the highest coefficient is estimated revenue followed by estimated occupancy, and number of reviews. Also parameters like maximum nights and minimum maximum nights also had relatively stronger positive relationships.

2. Features like maximum minimum nights and minimum nights have a negative coefficient, telling us that as minimum nights increase and the base listing price tends to decrease.

3. The results tell us that listings generating high estimated revenue and occupancy over the past year are priced higher. Also, shorter minimum night requirements allow hosts to charge higher nightly rates, whereas long term rentals or multi-listing hosts might price their listings more conservatively.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [17]:
# Select top 5 features with the highest absolute coefficients from our previous analysis
top_features = ['estimated_revenue_l365d', 'estimated_occupancy_l365d', 'number_of_reviews', 'maximum_nights_avg_ntm', 'minimum_maximum_nights']

# Filter features for training and testing splits
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# Initialize and fit the refined model
refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)

# Predict prices using the refined model
y_pred_refined = refined_model.predict(X_test_refined)

# Calculate metrics for the refined model
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

# Compare performance against the baseline model
print("=== Baseline Model (All 50 Features) ===")
print(f"MSE: {mse:.2f}")
print(f"R-squared (R2): {r2:.4f}\n")

print("=== Refined Model (Top 5 Features Only) ===")
print(f"Selected Features: {top_features}")
print(f"MSE: {mse_refined:.2f}")
print(f"R-squared (R2): {r2_refined:.4f}")

=== Baseline Model (All 50 Features) ===
MSE: 88037.67
R-squared (R2): 0.0786

=== Refined Model (Top 5 Features Only) ===
Selected Features: ['estimated_revenue_l365d', 'estimated_occupancy_l365d', 'number_of_reviews', 'maximum_nights_avg_ntm', 'minimum_maximum_nights']
MSE: 76805.70
R-squared (R2): 0.1961


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept estimated revenue, estimated occupancy, number of reviews, maximum nights, and minimum maximum nights because they demonstrated the highest absolute coefficients in the previous model, showing they have the strongest relationship with price.

2. Yes, performance improved dramatically. The R-squared score increased from 0.078 to 0.196, and the Mean Squared Error decreased from 88,037 to 76,805. This is because removing 45 noisy or weak features reduced overfitting and simplified the relationships.

3. I would recommend the newer refined model. It is much more accurate, simpler to explain to hosts, and requires less data collection and processing.

4. This connects to my learning outcomes because it highlights how feature selection is critical for predictive regression, bridging raw data modeling with practical and interpretable business logic. Especially as a finance major, that interpretable business logic is key and usually the missing piece when handling lots of data.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. Our model helped answer questions like "What are the key drivers behind listing prices, and can we predict daily rates using historical property performance and availability features?".
2. I would recommend to structure their listings to optimize occupancy and gather positive reviews. also, keep minimum night restrictions lower to maximize nightly rates, as higher minimum nights correlate with lower base prices.
3. I could remove extreme price outliers, incorporate non-linear factors and use gradient boosting.
4. Building and refining a regression model showed me how data cleansing, feature pruning, and model comparisons are utilized in the business world to build simple, actionable, and interpretable frameworks.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [18]:
!jupyter nbconvert --to html "carpenter_henry_assignment_11_regression.ipynb"

[NbConvertApp] Converting notebook carpenter_henry_assignment_11_regression.ipynb to html
[NbConvertApp] Writing 346581 bytes to carpenter_henry_assignment_11_regression.html
